<a href="https://colab.research.google.com/github/vihangasathsara-75/cm2604-cw-telco-churn-prediction/blob/main/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/vihangasathsara-75/cm2604-cw-telco-churn-prediction/refs/heads/main/data/raw/Telco-Churn.csv"
df = pd.read_csv(url)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
print(df.shape)

(7043, 20)


In [2]:
service_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]

for col in service_cols:
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})

for col in service_cols:
    print(col, df[col].unique())

MultipleLines ['No' 'Yes']
OnlineSecurity ['No' 'Yes']
OnlineBackup ['Yes' 'No']
DeviceProtection ['No' 'Yes']
TechSupport ['No' 'Yes']
StreamingTV ['No' 'Yes']
StreamingMovies ['No' 'Yes']


In [3]:
from sklearn.model_selection import train_test_split

y = df["Churn"].map({"No": 0, "Yes": 1})
X = df.drop(columns=["Churn"])
assert y.notnull().all()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print("Churn rate train:", round(y_train.mean(), 3), "test:", round(y_test.mean(), 3))

(5634, 19) (1409, 19)
Churn rate train: 0.265 test: 0.265


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = X.select_dtypes(include="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",   # keeps SeniorCitizen as it is (already 0/1)
    sparse_threshold=0,
)

In [ ]:
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print(X_train_p.shape, X_test_p.shape)
print(preprocessor.get_feature_names_out())
print("NaNs:", np.isnan(X_train_p).sum(), np.isnan(X_test_p).sum())